# Evaluate Benchmark

In [1]:
import sys
from pathlib import Path

project_root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in project_root_candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
from openai import OpenAI

from src.benchmark import BenchmarkEvaluator
from src.llm.llm import OpenAILLM

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("benchmark_evaluation")
logging.getLogger("httpx").setLevel(logging.WARNING)

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Index

In [2]:
from src.preprocessing import Reader, FixedCharChunker, Document, Page, Embedder
from src.indexing import DataStore, ChunkStore, FlatVectorStore, Index
from src.utils import AutoEDAIndex
from src.llm import OpenAILLM


DATA_STORE_DIR = PROJECT_ROOT / 'data/index/documents'
CHUNK_STORE_DIR = PROJECT_ROOT / 'data/index/chunks'
VECTOR_STORE_DIR = PROJECT_ROOT / 'data/index/vectors'
SQLITE_PATH = PROJECT_ROOT / 'data/index/db'

print(DATA_STORE_DIR)
print(CHUNK_STORE_DIR)
print(VECTOR_STORE_DIR)
print(SQLITE_PATH)

embedder = Embedder(
    device='mps', 
    model_name='ai-forever/FRIDA',
    type_handlers={
        'query': lambda text: f'search_query: {text}',
        'document': lambda text: f'search_document: {text}',
    }
)
dimensions = embedder.get_embeddings(['test']).shape[-1]

index = Index(
    datastore=DataStore(str(DATA_STORE_DIR), logger),
    vectorstore=FlatVectorStore(str(VECTOR_STORE_DIR), dimensions, logger),
    chunkstore=ChunkStore(str(CHUNK_STORE_DIR), logger),
    chunker=FixedCharChunker(chunk_size=500, overlap=50, logger=logger),
    embedder=embedder,
    reader=Reader(logger),
    sqlite_path=str(SQLITE_PATH),
    logger=logger
)
    
llm = OpenAILLM(client, model_name='gpt-5.4')


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: ai-forever/FRIDA


/Users/danilaandreev/Documents/HSE/degree/data/index/documents
/Users/danilaandreev/Documents/HSE/degree/data/index/chunks
/Users/danilaandreev/Documents/HSE/degree/data/index/vectors
/Users/danilaandreev/Documents/HSE/degree/data/index/db


Loading weights: 100%|██████████| 219/219 [00:00<00:00, 2260.47it/s, Materializing param=shared.weight]                                                    
INFO:sentence_transformers.SentenceTransformer:7 prompts are loaded, with the keys: ['paraphrase', 'search_query', 'search_document', 'categorize', 'categorize_sentiment', 'categorize_topic', 'categorize_entailment']
INFO:benchmark_evaluation:Index vectorstore loaded


## Baseline

In [3]:
from src.agents import V1Agent

In [4]:
judge_llm = OpenAILLM(client=client, model_name="gpt-5.4")
evaluator = BenchmarkEvaluator(judge_llm=judge_llm, logger=logger)

experiment_name = "rag_eval_v1"
benchmark_path = str(PROJECT_ROOT / "src/benchmark/results/generation_benchmark_v1_20260402_212121/benchmark.jsonl")

In [5]:
baseline_agent = V1Agent(index, llm, 10)

def answer_fn(question: str) -> str:
    return baseline_agent.answer(question)

In [6]:
result = evaluator.evaluate(
    benchmark_path=benchmark_path,
    answer_fn=answer_fn,
    experiment_name=experiment_name,
)

print(result["output_dir"])
result["metrics_df"]

Evaluating benchmark: 100%|██████████| 300/300 [1:08:56<00:00, 13.79s/it]

/Users/danilaandreev/Documents/HSE/degree/src/benchmark/results/evaluation_rag_eval_v1_20260403_202758


,question_type,metric,value,count
0,choice,score_accuracy,0.900000,50
1,comparison,score_attribute_accuracy,0.873333,50
2,comparison,score_hallucination_rate,0.073762,50
3,factual,score_accuracy,0.700000,50
4,multi_fact,score_f1,0.745567,50
5,multi_fact,score_hallucination_rate,0.264690,50
6,multi_fact,score_precision,0.735310,50
7,multi_fact,score_recall,0.788000,50
8,negative,score_abstention_accuracy,0.300000,50
9,negative,score_hallucination_rate,0.680000,50
